# Variational autoencoder (VAE) and clustering

In this notebook, you will implement and train a variational autoencoder (VAE) on the emoji dataset. You will then explore its latent space using dimensionality reduction techniques and clustering.

Run this notebook in Google Colab. Complete the five code blocks marked `Your code here`: the encoder/decoder, two PCA projections, k-means, and Leiden clustering. The data preparation, training loop, and plotting helpers are supplied. A GPU is optional for this small model. Downloads and the first UMAP call can take longer than subsequent runs.


## 0. Install dependencies

The cell below installs the required packages in Google Colab.


In [ ]:
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    %pip install -q emoji igraph leidenalg umap-learn uharfbuzz "pillow>=10" requests scikit-learn matplotlib scipy


In [ ]:
from collections import Counter
from io import BytesIO
from pathlib import Path
import random
import re
import tarfile
import textwrap

import emoji
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFont, features
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
import requests
import uharfbuzz as hb
import umap
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TWEMOJI_VERSION = "17.0.2"
TWEMOJI_URL = (
    "https://github.com/jdecked/twemoji/"
    f"archive/refs/tags/v{TWEMOJI_VERSION}.tar.gz"
)
TWEMOJI_CACHE_PATH = DATA_DIR / f"twemoji_v{TWEMOJI_VERSION}_72px.npz"

NOTO_VERSION = "2.051"
NOTO_FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-emoji/"
    f"v{NOTO_VERSION}/fonts/NotoColorEmoji.ttf"
)
NOTO_FONT_PATH = DATA_DIR / f"NotoColorEmoji-v{NOTO_VERSION}.ttf"
NOTO_CACHE_PATH = DATA_DIR / f"noto_v{NOTO_VERSION}_paired_v2_72px.npz"

DISPLAY_SIZE = 72
IMAGE_SIZE = 32
LATENT_DIM = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(torch.get_num_threads(), 2))
print("Device:", DEVICE)


## 1. Download and prepare paired emoji images

We download emoji artwork from [Twemoji](https://github.com/jdecked/twemoji) (CC BY 4.0) and render the same Unicode sequences with the [Noto Emoji font](https://github.com/googlefonts/noto-emoji) (OFL 1.1).

In [ ]:
def code_to_emoji(code: str) -> str:
    """Convert a Twemoji filename such as '1f600' into Unicode."""
    return "".join(chr(int(part, 16)) for part in code.split("-"))


def friendly_name(character: str) -> str:
    """Turn an emoji into a short readable name."""
    name = emoji.demojize(character, language="en").strip(":")
    return name.replace("_", " ")


def load_twemoji_72(cache_path: Path = TWEMOJI_CACHE_PATH):
    """Download Twemoji and keep its original 72×72 artwork."""
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        images = cached["images"]
        codes = cached["codes"].astype(str)
    else:
        print(f"Downloading Twemoji v{TWEMOJI_VERSION} ...")
        response = requests.get(TWEMOJI_URL, timeout=120)
        response.raise_for_status()

        images_list, codes_list = [], []
        with tarfile.open(fileobj=BytesIO(response.content), mode="r:gz") as archive:
            members = sorted(
                (
                    member for member in archive.getmembers()
                    if member.isfile() and "/assets/72x72/" in member.name
                    and member.name.endswith(".png")
                ),
                key=lambda member: member.name,
            )

            for member in members:
                source = archive.extractfile(member)
                if source is None:
                    continue
                with Image.open(source) as image:
                    rgba = image.convert("RGBA")
                    background = Image.new(
                        "RGBA", (DISPLAY_SIZE, DISPLAY_SIZE), "white"
                    )
                    background.alpha_composite(rgba)
                    images_list.append(
                        np.asarray(background.convert("RGB"), dtype=np.uint8)
                    )
                codes_list.append(Path(member.name).stem)

        images = np.stack(images_list)
        codes = np.asarray(codes_list)
        np.savez_compressed(cache_path, images=images, codes=codes)
        print("Saved cache:", cache_path)

    characters = np.asarray([code_to_emoji(code) for code in codes])
    names = np.asarray([friendly_name(char) for char in characters])
    return images, codes, characters, names


def render_noto_72(characters, codes, cache_path: Path = NOTO_CACHE_PATH):
    """Render the same Unicode sequences with Android's Noto Color Emoji."""
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        if np.array_equal(cached["codes"].astype(str), codes):
            return cached["images"]

    # NotoColorEmoji is a bitmap color font with a fixed 109 px strike.
    font = ImageFont.truetype(NOTO_FONT_PATH, size=109, layout_engine=ImageFont.Layout.RAQM)
    rendered = []
    for character in characters:
        canvas = Image.new("RGBA", (180, 180), (0, 0, 0, 0))
        draw = ImageDraw.Draw(canvas)
        draw.text((20, 20), character, font=font, embedded_color=True)
        content_box = canvas.getbbox()
        glyph = canvas.crop(content_box) if content_box else canvas
        glyph.thumbnail((66, 66), Image.Resampling.LANCZOS)
        output = Image.new("RGBA", (DISPLAY_SIZE, DISPLAY_SIZE), "white")
        position = (
            (DISPLAY_SIZE - glyph.width) // 2,
            (DISPLAY_SIZE - glyph.height) // 2,
        )
        output.alpha_composite(glyph, position)
        rendered.append(np.asarray(output.convert("RGB"), dtype=np.uint8))

    images = np.stack(rendered)
    np.savez_compressed(cache_path, images=images, codes=codes)
    print("Saved cache:", cache_path)
    return images


def resize_images(image_array, size=IMAGE_SIZE):
    return np.stack([
        np.asarray(
            Image.fromarray(image).resize((size, size), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
        for image in image_array
    ])


def supported_noto_sequences(characters):
    """Reject missing glyphs and sequences that split into multiple symbols."""
    if not features.check_feature("raqm"):
        raise RuntimeError("Emoji rendering requires a Pillow build with RAQM support.")
    if not NOTO_FONT_PATH.exists():
        print(f"Downloading Noto Color Emoji v{NOTO_VERSION} ...")
        response = requests.get(NOTO_FONT_URL, timeout=120)
        response.raise_for_status()
        NOTO_FONT_PATH.write_bytes(response.content)
    font = hb.Font(hb.Face(NOTO_FONT_PATH.read_bytes()))
    supported = []
    for character in characters:
        buffer = hb.Buffer()
        buffer.add_str(str(character))
        buffer.guess_segment_properties()
        hb.shape(font, buffer)
        missing = any(info.codepoint == 0 for info in buffer.glyph_infos)
        visible = sum(
            position.x_advance != 0 or position.y_advance != 0
            for position in buffer.glyph_positions
        )
        supported.append(not missing and visible == 1)
    return np.asarray(supported, dtype=bool)


display_images, emoji_codes, emoji_chars, emoji_names = load_twemoji_72()
paired_mask = supported_noto_sequences(emoji_chars)
print(f"Keeping {paired_mask.sum()} paired identities; excluding {(~paired_mask).sum()} unsupported sequences.")
display_images = display_images[paired_mask]
emoji_codes = emoji_codes[paired_mask]
emoji_chars = emoji_chars[paired_mask]
emoji_names = emoji_names[paired_mask]
noto_display_images = render_noto_72(emoji_chars, emoji_codes)
assert display_images.shape == noto_display_images.shape
assert display_images.shape[1:] == (DISPLAY_SIZE, DISPLAY_SIZE, 3)
model_images = resize_images(display_images)
noto_model_images = resize_images(noto_display_images)
platform_model_images = (model_images, noto_model_images)

print("Display images:", display_images.shape, display_images.dtype)
print("VAE input images:", model_images.shape, model_images.dtype)
print("Latent representation after training:", (len(model_images), LATENT_DIM))


Just explore some random emojis.

In [ ]:
def show_image_grid(grid_images, titles=None, columns=10, size=1.25, label_width=18):
    rows = int(np.ceil(len(grid_images) / columns))
    wrapped_titles = [
        textwrap.fill(str(title), width=label_width, break_long_words=False, break_on_hyphens=False)
        for title in titles
    ] if titles is not None else [""] * len(grid_images)
    # Reserve enough space in each row for its longest multiline label.
    row_heights = [
        size + 0.15 * max(len(title.splitlines()) for title in wrapped_titles[start:start + columns])
        for start in range(0, len(grid_images), columns)
    ]
    fig, axes = plt.subplots(
        rows, columns, figsize=(columns * size, sum(row_heights)), squeeze=False,
        gridspec_kw={"height_ratios": row_heights}, layout="constrained",
    )
    for axis in axes.flat:
        axis.axis("off")
    for i, image in enumerate(grid_images):
        axes.flat[i].imshow(image)
        if titles is not None:
            axes.flat[i].set_title(wrapped_titles[i], fontsize=7, pad=6)
    return fig


sample_rng = np.random.default_rng(SEED)
sample_indices = sample_rng.choice(len(display_images), size=30, replace=False)
show_image_grid(
    display_images[sample_indices],
    emoji_names[sample_indices],
    columns=10,
)
plt.show()


## 2. Add small image augmentations

We have relatively few distinct emoji identities, so we vary their appearance during training: choose a rendering platform and skin-tone variant, then apply small rotations and brightness/saturation changes. These variations do not create new independent identities. Each augmented image is both the model input and its reconstruction target.


In [ ]:
def augment_emoji(image_array: np.ndarray) -> np.ndarray:
    image = Image.fromarray(image_array)
    angle = np.random.uniform(-12, 12)
    image = image.rotate(
        angle,
        resample=Image.Resampling.BILINEAR,
        fillcolor=(255, 255, 255),
    )
    image = ImageEnhance.Color(image).enhance(np.random.uniform(0.75, 1.25))
    image = ImageEnhance.Brightness(image).enhance(np.random.uniform(0.88, 1.12))
    return np.asarray(image, dtype=np.uint8)


SKIN_TONE_POINTS = set(range(0x1F3FB, 0x1F400))


def without_skin_tones(code):
    points = [int(part, 16) for part in code.split("-")]
    return tuple(point for point in points if point not in SKIN_TONE_POINTS and point != 0xFE0F)


variants_by_base = {}
for index, code in enumerate(emoji_codes):
    variants_by_base.setdefault(without_skin_tones(code), []).append(index)
skin_variant_indices = [
    np.asarray(variants_by_base[without_skin_tones(code)])
    for code in emoji_codes
]
print("Sequences with skin-tone alternatives:", sum(len(x) > 1 for x in skin_variant_indices))


class EmojiDataset(Dataset):
    def __init__(self, platform_images, skin_variants=None, augment=False):
        self.platform_images = platform_images
        self.skin_variants = skin_variants
        self.augment = augment

    def __len__(self):
        return len(self.platform_images[0])

    def __getitem__(self, index):
        if self.augment:
            alternatives = self.skin_variants[index] if self.skin_variants is not None else [index]
            source_index = int(np.random.choice(alternatives))
            platform = np.random.randint(len(self.platform_images))
        else:
            source_index = index
            platform = 0

        image = self.platform_images[platform][source_index]
        if self.augment:
            image = augment_emoji(image)
        tensor = torch.from_numpy(image.copy()).float()
        return tensor.permute(2, 0, 1) / 255.0


example_index = int(np.flatnonzero(emoji_codes == "1f44d")[0])
example_variants = skin_variant_indices[example_index]
augmented = []
for _ in range(10):
    variant = int(np.random.choice(example_variants))
    platform = np.random.randint(2)
    source = (display_images, noto_display_images)[platform][variant]
    augmented.append(augment_emoji(source))
show_image_grid(
    [display_images[example_index], noto_display_images[example_index], *augmented],
    ["Twemoji", "Noto", *(["random platform/tone"] * 10)],
    columns=6,
)
plt.show()


## 3. Train a small VAE

VAE (Variational Autoencoders) will be covered in more depth during the next lecture. We will introduce a very breif idea here.

VAE is a method of learning representations of data based on two parts: an encoder and a decoder. Encoder is a model (typically a neural network) that maps input (in our case, emojis) to a latent representation which is a vector of lower dimensionality than the input. The decoder is a complementary model that tried to reconstruct the object from its latent representation.

VAE training consists of optimizing a so called reconstruction loss that measures how well the decoder can reconstruct the input image. The real loss also includes some additional terms that will be discussed during the next lecture.

As a result, the VAE learns a latent representation that captures the main features of the input data. For various reasons, discussed later during the course, this latent space contains a very rich geometry of the underlying data distribution and can be used for multiple downstream tasks. Multiple foundational models in different modalities are trained like that.

In this excercies you will train a very simple VAE on emoji images. Most of the code is already here, while you need to implement encoder and decoder architectures using PyTorch. An example architecture is provided below but you are more than welcome to experiment with your own designs.

In [ ]:
class LinearEmojiVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        input_dim = 3 * IMAGE_SIZE * IMAGE_SIZE

        self.mean = nn.Linear(64, latent_dim)
        self.log_variance = nn.Linear(64, latent_dim)

        self.encoder = None
        self.decoder = None

        # Define the encoder and decoder as small neural networks.
        # Architecture suggestion:
        # - Encoder: Flatten -> Linear(3*32*32, 256) -> ReLU -> Linear(256, 64) -> ReLU
        # - Decoder: Linear(latent_dim, 64) -> ReLU -> Linear(64, 256) -> ReLU -> Linear(256, 3*32*32) -> Sigmoid
        # But you are free (and encouraged) to experiment with architectures on your own.
        # Your code here.
        raise NotImplementedError("Complete this exercise.")

    def encode(self, x):
        hidden = self.encoder(x)
        return self.mean(hidden), self.log_variance(hidden)

    def reparameterize(self, mean, log_variance):
        standard_deviation = torch.exp(0.5 * log_variance)
        noise = torch.randn_like(standard_deviation)
        return mean + noise * standard_deviation

    def decode(self, z):
        reconstruction = self.decoder(z)
        return reconstruction.view(-1, 3, IMAGE_SIZE, IMAGE_SIZE)

    def forward(self, x):
        mean, log_variance = self.encode(x)
        z = self.reparameterize(mean, log_variance)
        return self.decode(z), mean, log_variance


# Weighted beta-VAE loss; both terms use means over their dimensions.
def vae_loss(reconstruction, original, mean, log_variance, beta=0.002):
    reconstruction_error = F.binary_cross_entropy(
        reconstruction, original, reduction="none"
    )
    pixel_weight = 1.0 + 2.0 * (1.0 - original)
    reconstruction_error = (pixel_weight * reconstruction_error).mean()

    kl_divergence = -0.5 * torch.mean(
        1 + log_variance - mean.square() - log_variance.exp()
    )
    return reconstruction_error + beta * kl_divergence, reconstruction_error, kl_divergence


Run training and check the cell below that shows the reconstruction for some emoji images. With one epoch, the reconstruction will likely be very rough, but you should see the model starting to capture the general shapes and colors of the emojis. Try to increase the number of epochs to improve the quality of the reconstructions. Watch how loss changes as you train for more epochs.

**Q1**: What is the best number of epochs? Which factors influence this choice?

**Q2** BONUS: Add a validation set and evaluate it after each epoch. Keep all skin-tone variants of the same base emoji, and both rendering platforms, in the same split. Otherwise augmentation can leak validation identities into training. Compare unaugmented training and validation losses under the same evaluation procedure. Is there evidence of overfitting?


In [ ]:
BATCH_SIZE = 256
EPOCHS = 1
LEARNING_RATE = 1e-3

train_dataset = EmojiDataset(
    platform_model_images,
    skin_variants=skin_variant_indices,
    augment=True,
)
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator,
)


@torch.no_grad()
def evaluate_vae(model, loader):
    """Estimate the same stochastic objective on unaugmented images."""
    model.eval()
    totals = np.zeros(3)
    devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
    # Fixed draws reduce Monte Carlo variation across epochs without changing
    # the random-number stream used for training.
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(SEED + 1)
        for batch in loader:
            batch = batch.to(DEVICE)
            values = vae_loss(*_loss_arguments(model, batch))
            totals += np.array([value.item() for value in values]) * len(batch)
    return totals / len(loader.dataset)


def _loss_arguments(model, batch):
    reconstruction, mean, log_variance = model(batch)
    return reconstruction, batch, mean, log_variance


def fit_vae(model, loader, epochs, evaluation_loaders=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    training_history = []
    evaluation_history = {name: [] for name in (evaluation_loaders or {})}
    for epoch in range(epochs):
        model.train()
        totals = np.zeros(3)
        for batch in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            values = vae_loss(*_loss_arguments(model, batch))
            loss, reconstruction_loss, kl_loss = values
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite training loss.")
            loss.backward()
            optimizer.step()
            totals += np.array([value.item() for value in values]) * len(batch)
        totals /= len(loader.dataset)
        training_history.append(totals)
        for name, evaluation_loader in (evaluation_loaders or {}).items():
            evaluation_history[name].append(evaluate_vae(model, evaluation_loader))
        print(f"epoch {epoch + 1:02d}/{epochs}: loss={totals[0]:.4f}, recon={totals[1]:.4f}, KL={totals[2]:.4f}")
    return np.asarray(training_history), {
        name: np.asarray(values) for name, values in evaluation_history.items()
    }


In [ ]:
model = LinearEmojiVAE().to(DEVICE)
history, _ = fit_vae(model, train_loader, EPOCHS)
assert np.isfinite(history).all()
epoch_axis = np.arange(1, EPOCHS + 1)
plt.plot(epoch_axis, history[:, 1], label="reconstruction")
plt.xlabel("epoch")
plt.ylabel("mean reconstruction loss")
plt.legend()
plt.show()


Here you can see `decode(mean(encode(emoji)))`

The small model may reconstruct colors and broad shapes while blurring details. Reconstruction encourages a useful representation but does not guarantee that the latent space captures semantic similarity.

It's perfectly normal for some reconstructions to be blurry or even be completely wrong, but you should see a general resemblance for some of them. This is also the case for the big foundational models. The goal is not to create a representation that perfectly reconstructs every detail, but to force the training process to capture the most important features of the data.


In [ ]:
@torch.no_grad()
def reconstruct(model, selected_images):
    model.eval()
    batch = torch.from_numpy(selected_images.copy()).float()
    batch = batch.permute(0, 3, 1, 2).to(DEVICE) / 255.0
    mean, _ = model.encode(batch)
    reconstruction = model.decode(mean)
    return (
        reconstruction.cpu()
        .permute(0, 2, 3, 1)
        .numpy()
        .clip(0, 1)
    )


check_indices = sample_rng.choice(len(model_images), size=10, replace=False)
reconstructed = reconstruct(model, model_images[check_indices])
comparison = []
comparison_titles = []
for original, decoded in zip(display_images[check_indices], reconstructed):
    comparison.extend([original, decoded])
    comparison_titles.extend(["original", "VAE"])
show_image_grid(comparison, comparison_titles, columns=10)
plt.show()


Compute deterministic encoder means for all emoji in both platforms. Their average is used for clustering. Averaging can reduce a platform-specific shift, but does not guarantee platform invariance; section 4b inspects the separate representations.


In [ ]:
@torch.no_grad()
def encode_all(model, image_array, batch_size=512):
    """Return the deterministic VAE means for one platform."""
    model.eval()
    loader = DataLoader(
        EmojiDataset((image_array,), augment=False),
        batch_size=batch_size,
        shuffle=False,
    )
    parts = []
    for batch in loader:
        mean, _ = model.encode(batch.to(DEVICE))
        parts.append(mean.cpu().numpy())
    return np.concatenate(parts)


latent_twemoji = encode_all(model, model_images)
latent_noto = encode_all(model, noto_model_images)
latent = (latent_twemoji + latent_noto) / 2
print("Latent matrix:", latent.shape)
assert latent.shape == (len(model_images), LATENT_DIM)
assert np.isfinite(latent).all()
print("Finite values:", np.isfinite(latent).all())


## 4. Inspect the latent space with PCA

**Q3**: Use PCA to reduce the latent space to 2D. Country/subdivision flags and a heuristic selection of animal names are highlighted. Do flags or animals form visible groups? Why might a model trained to reconstruct pixels group images by color or shape rather than meaning?

**Q4** BONUS: Compare with a seeded UMAP projection. Does it make these groups easier to see? Do not interpret gaps in the UMAP plot as quantitative distances between semantic categories.


In [ ]:
def code_points(code):
    return [int(part, 16) for part in code.split("-")]


ANIMAL_TERMS = {
    "monkey", "ape", "orangutan", "dog", "poodle", "wolf",
    "fox", "raccoon", "cat", "lion", "tiger", "leopard",
    "horse", "moose", "donkey", "unicorn", "zebra", "deer",
    "bison", "cow", "ox", "buffalo", "pig", "boar", "ram",
    "ewe", "goat", "camel", "llama", "giraffe", "elephant",
    "mammoth", "rhinoceros", "hippopotamus", "mouse", "rat",
    "hamster", "rabbit", "chipmunk", "beaver", "hedgehog",
    "bat", "bear", "koala", "panda", "sloth", "otter",
    "skunk", "kangaroo", "badger", "turkey", "chicken",
    "rooster", "chick", "bird", "penguin", "dove", "eagle",
    "duck", "swan", "owl", "dodo", "flamingo", "peacock",
    "parrot", "goose", "phoenix", "frog", "crocodile",
    "turtle", "lizard", "snake", "dragon", "sauropod", "t-rex",
    "whale", "dolphin", "seal", "fish", "shark", "octopus",
    "jellyfish", "snail", "butterfly", "bug", "ant", "honeybee",
    "beetle", "cricket", "cockroach", "spider", "scorpion",
    "mosquito", "fly", "worm",
}


def is_flag_code(code):
    points = code_points(code)
    regional_flag = (
        len(points) == 2
        and all(0x1F1E6 <= point <= 0x1F1FF for point in points)
    )
    subdivision_flag = (
        len(points) > 2 and points[0] == 0x1F3F4 and points[-1] == 0xE007F
        and all(0xE0061 <= point <= 0xE007A for point in points[1:-1])
    )
    return regional_flag or subdivision_flag


def is_animal_name(name):
    if name in {"hot dog", "cricket game"}:
        return False
    return any(
        re.search(rf"\b{re.escape(term)}\b", name)
        for term in ANIMAL_TERMS
    )


flag_mask = np.asarray([is_flag_code(code) for code in emoji_codes])
animal_mask = np.asarray([is_animal_name(name) for name in emoji_names])

# Project latent into 2D space.
# Your code here.
raise NotImplementedError("Complete this exercise.")

fig, axis = plt.subplots(figsize=(8, 6))
axis.scatter(
    latent_pca[:, 0], latent_pca[:, 1],
    s=8, color="lightgray", alpha=0.35, label="other emoji",
)
axis.scatter(
    latent_pca[flag_mask, 0], latent_pca[flag_mask, 1],
    s=18, color="#377eb8", alpha=0.85, label=f"flags ({flag_mask.sum()})",
)
axis.scatter(
    latent_pca[animal_mask, 0], latent_pca[animal_mask, 1],
    s=24, color="#4daf4a", alpha=0.90, label=f"animals ({animal_mask.sum()})",
)
axis.set(title=f"PCA of the {LATENT_DIM}-D VAE means", xlabel="PC 1", ylabel="PC 2")
axis.legend()
plt.show()


## 4b. Platform batch effects

Here, we render some emojis in two styles (Twemoji and Noto Emoji) and visualize the latent space using PCA. Compare the separation between platform centroids with the spread within each platform. Does rendering style have a strong effect for this selected set of facial emoji? The result depends on training and the chosen identities; it need not hold for every emoji. Font cropping and scaling also contribute to the rendering difference.


In [ ]:
# Restrict identity variation so platform style can dominate the first PCs.
first_code_point = np.asarray([code_points(code)[0] for code in emoji_codes])
batch_indices = np.flatnonzero(
    (first_code_point >= 0x1F600) & (first_code_point <= 0x1F644)
)
batch_size = len(batch_indices)

latent_batches = np.concatenate([
    latent_twemoji[batch_indices],
    latent_noto[batch_indices],
])
platform_labels = np.repeat(
    ["Twemoji", "Noto / Android"], batch_size
)


PAIR_EXAMPLES = 8
batch_rng = np.random.default_rng(SEED)
pair_example_indices = batch_rng.choice(
    batch_indices, size=PAIR_EXAMPLES, replace=False
)
fig, axes = plt.subplots(2, PAIR_EXAMPLES, figsize=(12, 3.2), squeeze=False)
for column, index in enumerate(pair_example_indices):
    axes[0, column].imshow(display_images[index])
    axes[1, column].imshow(noto_display_images[index])
    axes[0, column].set_title(emoji_names[index][:20], fontsize=7)
for axis in axes.flat:
    axis.axis("off")
fig.text(0.01, 0.66, "Twemoji", rotation=90, va="center", fontsize=9)
fig.text(0.01, 0.27, "Noto / Android", rotation=90, va="center", fontsize=9)
fig.suptitle("Corresponding platform pairs: same Unicode, different rendering")
plt.tight_layout()
plt.show()


def plot_platform_pca(features, axis, title):
    # Project features into 2D space.
    # Your code here.
    raise NotImplementedError("Complete this exercise.")
    colors = {"Twemoji": "#1f77b4", "Noto / Android": "#ff7f0e"}

    for platform in colors:
        mask = platform_labels == platform
        batch_coordinates = coordinates[mask]
        centroid = batch_coordinates.mean(axis=0)
        axis.scatter(
            batch_coordinates[:, 0], batch_coordinates[:, 1],
            s=22, alpha=0.58, color=colors[platform], label=platform,
        )
        axis.scatter(
            *centroid, s=170, marker="X", color=colors[platform],
            edgecolor="white", linewidth=1.2,
        )

    axis.set(
        title=title,
        xlabel="PC 1", ylabel="PC 2",
    )
    axis.legend()


fig, axis = plt.subplots(figsize=(8, 6))
plot_platform_pca(latent_batches, axis, f"PCA of {LATENT_DIM}-D encoder means")
fig.suptitle(
    f"Platform batch effect for {batch_size} paired facial emoji identities"
)
plt.tight_layout()
plt.show()


## 5. K-means clustering

Start with a simpler clustering method: k-means. It alternates between assigning points to their nearest cluster center and updating each center to the mean of its assigned points. You choose the number of clusters, `k`.

Implement `kmeans_from_latent` using [sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html). Fit on the full latent vectors, using `random_state=seed` and `n_init=10`, and return one cluster label per emoji. The code below starts with `k=12` and displays examples from each cluster.

**Q5**: Try several values of `k`. Which groups are easy to explain by meaning, color, or shape? What happens when `k` is too small or too large? After the next section, compare k-means with Leiden on the same latent vectors.


In [ ]:
STOP_WORDS = {
    "with", "and", "of", "the", "on", "in", "a", "an",
    "medium", "light", "dark", "skin", "tone",
    "person", "woman", "man",
}


def frequent_name_words(indices, number=3):
    tokens = []
    for name in emoji_names[indices]:
        tokens.extend(
            word for word in re.findall(r"[a-z]+", name.lower())
            if word not in STOP_WORDS and len(word) > 2
        )
    return ", ".join(word for word, _ in Counter(tokens).most_common(number))


def show_cluster_examples(
    cluster_labels, clusters_to_show=12, examples_per_cluster=8, seed=SEED,
    title="Randomly selected clusters",
):
    all_clusters = np.unique(cluster_labels)
    rng = np.random.default_rng(seed)
    clusters = rng.choice(
        all_clusters,
        size=min(clusters_to_show, len(all_clusters)),
        replace=False,
    )
    clusters = sorted(
        clusters, key=lambda value: -(cluster_labels == value).sum()
    )
    fig, axes = plt.subplots(
        len(clusters),
        examples_per_cluster,
        figsize=(examples_per_cluster * 1.25, len(clusters) * 1.3),
        squeeze=False,
    )

    for row, cluster in enumerate(clusters):
        members = np.flatnonzero(cluster_labels == cluster)
        selected = rng.choice(
            members,
            size=min(examples_per_cluster, len(members)),
            replace=False,
        )
        summary = frequent_name_words(members)
        axes[row, 0].set_ylabel(
            f"C{cluster} · n={len(members)}\n{summary}",
            fontsize=8,
            rotation=0,
            ha="right",
            va="center",
            labelpad=42,
        )
        for column, index in enumerate(selected):
            axes[row, column].imshow(display_images[index])
        for column in range(examples_per_cluster):
            axes[row, column].set_xticks([])
            axes[row, column].set_yticks([])
            axes[row, column].spines[:].set_visible(False)

    fig.suptitle(
        title,
        y=1.002,
    )
    plt.tight_layout()
    return fig


In [ ]:
def kmeans_from_latent(vectors, n_clusters, seed=SEED):
    """Return one k-means cluster label per latent vector."""
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


N_CLUSTERS = 12
kmeans_labels = kmeans_from_latent(latent, N_CLUSTERS)
assert kmeans_labels.shape == (len(latent),)
print("Cluster sizes:", sorted(Counter(kmeans_labels).values(), reverse=True))
show_cluster_examples(kmeans_labels, title=f"K-means clusters (k={N_CLUSTERS})")
plt.show()


## 6. Leiden clustering

Use Leiden to cluster emoji based on their learned latent representations. The supplied helper constructs an undirected, weighted nearest-neighbor graph once. Use [leidenalg.find_partition](https://leidenalg.readthedocs.io/en/stable/reference.html#leidenalg.find_partition) with `RBConfigurationVertexPartition`, the graph's `weight` edge attribute, the supplied resolution, and seed.

Run the cell and inspect how the number and size of clusters change with resolution. Pick a resolution and inspect examples in the next cell. Which clusters reflect meaning, color, or shape? Which resolution is useful for your question? A larger resolution generally favors smaller communities, but there is no universally best value or guaranteed semantic interpretation.


In [ ]:
def build_knn_graph(vectors, neighbors=15):
    vectors = np.asarray(vectors, dtype=float)
    if vectors.ndim != 2 or len(vectors) < 2 or not np.isfinite(vectors).all():
        raise ValueError("Expected at least two finite feature vectors.")
    if not isinstance(neighbors, (int, np.integer)) or neighbors < 1:
        raise ValueError("neighbors must be a positive integer.")
    k = min(neighbors, len(vectors) - 1)
    search = NearestNeighbors(n_neighbors=k).fit(vectors)
    # With no query matrix, sklearn excludes each sample itself, even with ties.
    distances, indices = search.kneighbors()
    positive_distances = distances[distances > 0]
    scale = np.median(positive_distances) if positive_distances.size else 1.0
    edge_weights = {}
    for source in range(len(vectors)):
        for target, distance in zip(indices[source], distances[source]):
            edge = tuple(sorted((source, int(target))))
            weight = float(np.exp(-min(distance / scale, 700)))
            edge_weights[edge] = max(weight, edge_weights.get(edge, 0.0))
    graph = ig.Graph(n=len(vectors), edges=list(edge_weights), directed=False)
    graph.es["weight"] = list(edge_weights.values())
    assert not any(graph.is_loop())
    return graph


def leiden_from_graph(graph, resolution=0.7, seed=SEED):
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


latent_graph = build_knn_graph(latent)
candidate_resolutions = [0.40, 0.70, 1.00, 1.30, 1.60, 1.90, 2.20, 2.50]
candidate_clusters = {
    resolution: leiden_from_graph(latent_graph, resolution=resolution)
    for resolution in candidate_resolutions
}
for resolution, labels_at_resolution in candidate_clusters.items():
    sizes = Counter(labels_at_resolution).values()
    print(f"resolution={resolution:.2f}: {len(np.unique(labels_at_resolution))} clusters, largest={max(sizes)}")

CHOSEN_RESOLUTION = 2.20
cluster_id = candidate_clusters[CHOSEN_RESOLUTION]
counts = Counter(cluster_id)
print("Largest chosen clusters:", sorted(counts.values(), reverse=True)[:10])


In [ ]:
show_cluster_examples(cluster_id, title="Randomly selected Leiden clusters")
plt.show()


## 7. Hierarchical clustering of a chosen emoji set

Use the supplied Ward-clustering helper to build a dendrogram, starting with the provided faces. Do happy and sad faces separate? Try `ANIMALS` and then your own selection. Can you explain the merges using visual features? Find a selection for which the grouping does not agree with your semantic expectations.

Ward clustering greedily merges groups to minimize the increase in within-cluster squared Euclidean dispersion. Read the dendrogram from short merges toward its final merge; a semantic split is not guaranteed. Its heights are Ward linkage distances, not probabilities or direct distances between every pair of images.


In [ ]:
EMOJIS_TO_CLUSTER = [
    "😀", "😃", "😄",
    "😔", "😞", "🙁", "☹️", "😟", "😕",
]

ANIMALS = [
    "🐶", "🐱", "🐭", "🐹", "🐰", "🦊", "🐻", "🐼", "🐨",
    "🦁", "🐯", "🐮", "🐷", "🐸", "🐵",
]

def normalize_emoji_character(character):
    # U+FE0F asks for emoji presentation but does not change identity here.
    return character.replace("\ufe0f", "")


emoji_to_index = {
    normalize_emoji_character(character): index
    for index, character in enumerate(emoji_chars)
}
def plot_emoji_dendrogram(selection, title="Ward clustering of selected emoji"):
    normalized_selection = [
        normalize_emoji_character(character) for character in selection
    ]
    missing = [
        character for character, normalized in zip(selection, normalized_selection)
        if normalized not in emoji_to_index
    ]

    if missing:
        raise ValueError(f"Emoji not available in the paired dataset: {missing}")
    if len(normalized_selection) < 2:
        raise ValueError("Select at least two emoji.")
    if len(set(normalized_selection)) != len(normalized_selection):
        raise ValueError("Choose distinct emoji after removing variation selectors.")

    selected_indices = np.asarray([
        emoji_to_index[character] for character in normalized_selection
    ])

    selected_vectors = latent[selected_indices]
    tree = linkage(selected_vectors, method="ward")

    fig, axis = plt.subplots(figsize=(10, max(5, 0.7 * len(selected_indices))))
    tree_info = dendrogram(
        tree,
        orientation="right",
        no_labels=True,
        link_color_func=lambda _: "#4c78a8",
        ax=axis,
    )

    maximum_distance = max(float(tree[:, 2].max()), np.finfo(float).eps)
    axis.set_xlim(-0.20 * maximum_distance, 1.03 * maximum_distance)
    for leaf_y, sample_position in zip(
        np.arange(5, 10 * len(selected_indices), 10), tree_info["leaves"]
    ):
        image_index = selected_indices[sample_position]
        thumbnail = OffsetImage(display_images[image_index], zoom=0.48)
        annotation = AnnotationBbox(
            thumbnail,
            (-0.09 * maximum_distance, leaf_y),
            frameon=True,
            pad=0.08,
            bboxprops={"edgecolor": "#555555", "linewidth": 1.5},
            annotation_clip=False,
        )
        axis.add_artist(annotation)

    axis.set_yticks([])
    axis.set_title(title)
    axis.set_xlabel(f"Ward linkage distance in {LATENT_DIM}-D VAE space")
    plt.tight_layout()
    return tree, selected_indices


face_tree, face_indices = plot_emoji_dendrogram(EMOJIS_TO_CLUSTER)
plt.show()
